# E-Commerce Lakehouse: Bronze Payments

## Ingest and validate source payments

Read the original payment CSV with source values preserved as strings.
Add ingestion metadata, save a managed Bronze Delta table, and
compare source and saved row counts.

## Ingest and refresh Bronze payments

Read the complete payments CSV with source values preserved as
strings. Add ingestion metadata and verify 103,886 source records
before replacing the Bronze table. Confirm the saved row count.

In [ ]:
from pyspark.sql.functions import current_timestamp

raw_payments_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/ecommerce/raw_files/olist_order_payments_dataset.csv")
)

source_count = raw_payments_df.count()
print("Source payment rows:", source_count)

if source_count != 103886:
    raise ValueError(
        f"Expected 103,886 payment records; found {source_count}. "
        "Stopped before replacing the table."
    )

bronze_payments_df = raw_payments_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

table_name = "workspace.ecommerce.bronze_payments"

(
    bronze_payments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_payments_df = spark.table(table_name)
saved_count = saved_payments_df.count()

if saved_count != source_count:
    raise ValueError("Saved Bronze payment count does not match the source.")

print("Bronze payments refreshed:", saved_count)

In [ ]:
print("Source payment rows:", raw_payments_df.count())
raw_payments_df.printSchema()